# Synchronized audio-video generation with LTX-2.3 and OpenVINO(TM)

[LTX-2.3](https://huggingface.co/diffusers/LTX-2.3-Diffusers) is a diffusion transformer model that generates video and synchronized audio in a single pipeline. The prompt can describe both the visual scene and its soundtrack, including ambient sounds, music, and speech.

This tutorial shows how to export LTX-2.3 to OpenVINO Intermediate Representation with [Optimum Intel](https://huggingface.co/docs/optimum/intel/index) and run one of two workflows:

- **Text-to-audio-video** generates video and audio from a text prompt.
- **Image-to-audio-video** uses an initial image together with a text prompt.

A single image-to-video export is reused by both workflows. It contains the common LTX-2.3 components and the VAE encoder needed only for image conditioning; the text-to-video pipeline ignores that extra component.

> **Resource notice:** LTX-2.3 is a large model. Download, export, and inference require substantial disk space and memory and may take a long time, especially on CPU.

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Convert the model to OpenVINO IR](#Convert-the-model-to-OpenVINO-IR)
- [Run audio-video generation](#Run-audio-video-generation)
- [Interactive inference](#Interactive-inference)
- [Limitations](#Limitations)

### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/ltx2.3-audio-video/ltx2.3-audio-video.ipynb" />

## Prerequisites
[back to top](#Table-of-content:)

LTX-2.3 support is currently available on the Optimum Intel `main` branch. The model requires Diffusers 0.40 and Transformers 5.10. Accept the model terms on Hugging Face before running the authentication cell.

In [ ]:
%pip install -qU "torch==2.10" "torchvision==0.25" "diffusers==0.40.0" "transformers==5.10.4" "accelerate" "sentencepiece" "protobuf>=3.20" "huggingface-hub" "av" "gradio>=6.0" "nncf>=2.19" --extra-index-url https://download.pytorch.org/whl/cpu
%pip install -qU "openvino>=2026.0" "openvino-tokenizers>=2026.0"
%pip install -qU "git+https://github.com/huggingface/optimum-intel.git@main" --extra-index-url https://download.pytorch.org/whl/cpu

In [ ]:
from pathlib import Path

import requests
from huggingface_hub import get_token, notebook_login

for file_name, url in {
    "notebook_utils.py": "https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py",
    "cmd_helper.py": "https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/cmd_helper.py",
    "gradio_helper.py": "https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/notebooks/ltx2.3-audio-video/gradio_helper.py",
}.items():
    if not Path(file_name).exists():
        response = requests.get(url)
        response.raise_for_status()
        Path(file_name).write_text(response.text, encoding="utf-8")

if get_token() is None:
    notebook_login()

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("ltx2.3-audio-video.ipynb")

## Convert the model to OpenVINO IR
[back to top](#Table-of-content:)

Optimum Intel's LTX-2.3 exporter produces the text encoder, connectors, transformer, video VAE encoder and decoder, audio VAE decoder, and vocoder for either task. Exporting the image-to-video task once creates a superset that can be loaded by both OpenVINO pipeline classes.

Use FP16 as the baseline OpenVINO weight format. INT8 and INT4 reduce model size, but the Optimum Intel pull request notes that INT8 may underperform, and INT4 quality has not been validated in this notebook.

In [ ]:
import ipywidgets as widgets

weight_format_widget = widgets.Dropdown(
    options=[
        ("FP16", "fp16"),
        ("INT8 (experimental; may reduce quality)", "int8"),
        ("INT4 (experimental; quality not validated)", "int4"),
    ],
    value="fp16",
    description="Weights:",
    style={"description_width": "initial"},
)

weight_format_widget

In [ ]:
from cmd_helper import optimum_cli

model_id = "diffusers/LTX-2.3-Diffusers"
weight_format = weight_format_widget.value
model_path = Path("ltx-2.3-ov") / weight_format.upper()

additional_args = {"task": "image-to-video", "weight-format": weight_format}

if not model_path.exists():
    optimum_cli(model_id, model_path, additional_args=additional_args)

print(f"Using {weight_format.upper()} model from {model_path}")

## Run audio-video generation
[back to top](#Table-of-content:)

Select the generation workflow and an OpenVINO device. Both workflows load the same exported model directory. NPU is excluded until the complete LTX-2.3 pipeline, including its audio components, is validated on that device.

In [ ]:
from notebook_utils import device_widget

generation_mode_widget = widgets.Dropdown(
    options=[
        ("Text-to-audio-video", "text-to-video"),
        ("Image-to-audio-video", "image-to-video"),
    ],
    value="text-to-video",
    description="Mode:",
)
device = device_widget(exclude=["NPU"])

display(generation_mode_widget, device)

In [ ]:
from optimum.intel.openvino import OVLTX2ImageToVideoPipeline, OVLTX2Pipeline

generation_mode = generation_mode_widget.value
pipeline_class = OVLTX2Pipeline if generation_mode == "text-to-video" else OVLTX2ImageToVideoPipeline
pipe = pipeline_class.from_pretrained(model_path, device=device.value)

The default settings below follow the LTX-2.3 reference example. The prompt should describe both what is visible and what should be heard. For image-to-audio-video, provide a URL or local path to the conditioning image.

In [ ]:
prompt_widget = widgets.Textarea(
    value="A cat stretches lazily on a sunny windowsill, purring softly, while birds chirp outside.",
    description="Prompt:",
    layout=widgets.Layout(width="90%", height="100px"),
    style={"description_width": "initial"},
)
image_widget = widgets.Text(
    value="https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/cosmos/cosmos-video2world-input.jpg",
    description="Image URL/path:",
    layout=widgets.Layout(width="90%"),
    style={"description_width": "initial"},
)

display(prompt_widget)
if generation_mode == "image-to-video":
    display(image_widget)

In [ ]:
import torch
from diffusers.pipelines.ltx2.utils import DEFAULT_NEGATIVE_PROMPT
from diffusers.utils import encode_video, load_image

frame_rate = 24.0
generation_args = {
    "prompt": prompt_widget.value,
    "negative_prompt": DEFAULT_NEGATIVE_PROMPT,
    "width": 768,
    "height": 512,
    "num_frames": 121,
    "frame_rate": frame_rate,
    "num_inference_steps": 30,
    "guidance_scale": 3.0,
    "generator": torch.Generator("cpu").manual_seed(42),
    "output_type": "np",
    "return_dict": False,
}
if generation_mode == "image-to-video":
    generation_args["image"] = load_image(image_widget.value)
    # Disable H.264 recompression of the conditioning image and preserve multi-batch compatibility.
    generation_args["image_crf"] = 0

video, audio = pipe(**generation_args)
output_path = "ltx2.3_output.mp4"
encode_video(
    video[0],
    fps=frame_rate,
    audio=audio[0].float().cpu(),
    audio_sample_rate=pipe.vocoder.config.output_sampling_rate,
    output_path=output_path,
)

In [ ]:
from IPython.display import Video

Video(output_path, embed=True)

### Guidance controls

LTX-2.3 supports classifier-free guidance, spatio-temporal guidance (STG), and modality-isolation guidance. STG and modality guidance add transformer passes and increase generation time. The exported pipeline does not support `use_cross_timestep=True`, and an explicit STG `perturbation_mask` must be uniform across a batch. The reference call above uses the pipeline defaults and only sets CFG through `guidance_scale`.

## Interactive inference
[back to top](#Table-of-content:)

The demo exposes the input image only for the image-to-audio-video workflow and returns an MP4 containing the generated audio track.

In [ ]:
from gradio_helper import make_demo

demo = make_demo(pipe, generation_mode)

try:
    demo.launch(debug=True)
except Exception:
    demo.launch(share=True, debug=True)
# If you are launching remotely, specify server_name and server_port.

## Limitations
[back to top](#Table-of-content:)

- The model repository is gated. Accept its terms and authenticate with a token that can read gated repositories.
- Export and inference are resource intensive. CPU inference can take a long time.
- FP16 is the baseline format. INT8 may noticeably reduce generation quality according to the Optimum Intel pull request, while INT4 has not been quality-validated in this notebook.
- Text-to-video and image-to-video reuse one image-to-video export. Rerun the mode and pipeline-loading cells after changing the workflow; conversion is not repeated.
- The OpenVINO integration covers the single-stage `LTX2Pipeline` and `LTX2ImageToVideoPipeline`. The Diffusers two-stage latent upsampler and arbitrary condition pipeline are outside this tutorial.
- NPU execution is not exposed until the full audio-video pipeline has been validated on that device.